In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "water_potability_balanced.csv"

In [5]:
df = pd.read_csv(RAW_FILE)

print("SC11 WATER QUALITY DATA INSPECTION")
print("-" * 45)

print(f"\nDataset shape: {df.shape[0]} rows, {df.shape[1]} columns")

SC11 WATER QUALITY DATA INSPECTION
---------------------------------------------

Dataset shape: 2556 rows, 10 columns


In [6]:
print("\nColumn names:")
print(df.columns.tolist())


Column names:
['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'Potability']


In [7]:
print("\nMissing values per column:")
print(df.isnull().sum())


Missing values per column:
ph                 372
Hardness             0
Solids               0
Chloramines          0
Sulfate            609
Conductivity         0
Organic_carbon       0
Trihalomethanes    129
Turbidity            0
Potability           0
dtype: int64


In [9]:
print(f"\nDuplicate rows: {df.duplicated().sum()}")

print("\nPotability class distribution:")
print(df["Potability"].value_counts().sort_index())


Duplicate rows: 0

Potability class distribution:
Potability
0    1278
1    1278
Name: count, dtype: int64


In [10]:
print("\nFirst five rows:")
display(df.head())


First five rows:


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,6.888843,188.920671,27368.014339,8.003187,355.478563,465.346979,12.377083,75.112494,4.512332,1
1,9.135694,194.815618,9496.910354,6.185925,345.714206,562.771467,13.676435,74.424166,3.281802,0
2,7.920795,173.357151,13454.634777,6.947895,356.768119,471.368695,16.477840,68.000508,3.374327,0
3,6.630122,192.521703,36269.701317,4.973866,277.499297,320.470853,14.786347,80.980213,3.352765,1
4,6.128426,141.492410,9615.832710,7.627919,277.951704,415.008737,16.435843,50.693927,4.633824,0


In [11]:
# STEP 2B: Clean the dataset without changing the original raw file

feature_columns = [
    "ph",
    "Hardness",
    "Solids",
    "Chloramines",
    "Sulfate",
    "Conductivity",
    "Organic_carbon",
    "Trihalomethanes",
    "Turbidity",
]

# Work on a copy. The original df and CSV in data/raw remain untouched.
clean_df = df.copy()

# Record the condition of the dataset before cleaning.
missing_before = clean_df.isna().sum()
duplicates_before = int(clean_df.duplicated().sum())

In [12]:
# Fill missing numeric values with the median of that column.
# Median is less affected by unusually high or low measurements.
clean_df[feature_columns] = clean_df[feature_columns].fillna(
    clean_df[feature_columns].median()
)

# Remove only genuine fully repeated rows.
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

In [14]:
print("DATA CLEANING SUMMARY")
print("-" * 45)
print(f"Rows before cleaning: {len(df)}")
print(f"Rows after cleaning:  {len(clean_df)}")
print(f"Duplicate rows removed: {duplicates_before}")

DATA CLEANING SUMMARY
---------------------------------------------
Rows before cleaning: 2556
Rows after cleaning:  2556
Duplicate rows removed: 0


In [15]:
print("\nMissing values before cleaning:")
print(missing_before)


Missing values before cleaning:
ph                 372
Hardness             0
Solids               0
Chloramines          0
Sulfate            609
Conductivity         0
Organic_carbon       0
Trihalomethanes    129
Turbidity            0
Potability           0
dtype: int64


In [16]:
print("\nMissing values after cleaning:")
print(clean_df.isna().sum())


Missing values after cleaning:
ph                 0
Hardness           0
Solids             0
Chloramines        0
Sulfate            0
Conductivity       0
Organic_carbon     0
Trihalomethanes    0
Turbidity          0
Potability         0
dtype: int64


In [17]:
print("\nFirst five cleaned rows:")
display(clean_df.head())


First five cleaned rows:


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,6.888843,188.920671,27368.014339,8.003187,355.478563,465.346979,12.377083,75.112494,4.512332,1
1,9.135694,194.815618,9496.910354,6.185925,345.714206,562.771467,13.676435,74.424166,3.281802,0
2,7.920795,173.357151,13454.634777,6.947895,356.768119,471.368695,16.477840,68.000508,3.374327,0
3,6.630122,192.521703,36269.701317,4.973866,277.499297,320.470853,14.786347,80.980213,3.352765,1
4,6.128426,141.492410,9615.832710,7.627919,277.951704,415.008737,16.435843,50.693927,4.633824,0


In [18]:
# STEP 2C: Validate the cleaned dataset

required_columns = [
    "ph",
    "Hardness",
    "Solids",
    "Chloramines",
    "Sulfate",
    "Conductivity",
    "Organic_carbon",
    "Trihalomethanes",
    "Turbidity",
    "Potability",
]

numeric_columns = [
    "ph",
    "Hardness",
    "Solids",
    "Chloramines",
    "Sulfate",
    "Conductivity",
    "Organic_carbon",
    "Trihalomethanes",
    "Turbidity",
]

In [19]:
validation_results = {
    "Missing required columns": len(
        [column for column in required_columns if column not in clean_df.columns]
    ),
    "Missing values remaining": int(clean_df[required_columns].isna().sum().sum()),
    "pH values outside 0 to 14": int((~clean_df["ph"].between(0, 14)).sum()),
    "Negative values in measurement columns": int(
        (clean_df[numeric_columns] < 0).sum().sum()
    ),
    "Invalid Potability labels": int(
        (~clean_df["Potability"].isin([0, 1])).sum()
    ),
    "Duplicate rows remaining": int(clean_df.duplicated().sum()),
}

print("CLEANED DATA VALIDATION")
print("-" * 45)



CLEANED DATA VALIDATION
---------------------------------------------


In [20]:
for check, count in validation_results.items():
    print(f"{check}: {count}")

Missing required columns: 0
Missing values remaining: 0
pH values outside 0 to 14: 0
Negative values in measurement columns: 0
Invalid Potability labels: 0
Duplicate rows remaining: 0


In [21]:
if sum(validation_results.values()) == 0:
    print("\nVALIDATION PASSED: Cleaned data is ready for splitting.")
else:
    print("\nVALIDATION NEEDS ATTENTION: Fix the checks with non-zero values.")


VALIDATION PASSED: Cleaned data is ready for splitting.


In [22]:
print("\nData types:")
print(clean_df.dtypes)

print("\nPotability class distribution after cleaning:")
print(clean_df["Potability"].value_counts().sort_index())


Data types:
ph                 float64
Hardness           float64
Solids             float64
Chloramines        float64
Sulfate            float64
Conductivity       float64
Organic_carbon     float64
Trihalomethanes    float64
Turbidity          float64
Potability           int64
dtype: object

Potability class distribution after cleaning:
Potability
0    1278
1    1278
Name: count, dtype: int64


In [24]:
# STEP 2D: Create reproducible train, validation and test splits

from pathlib import Path

SEED = 42
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

def create_stratified_splits(dataframe):
    train_parts = []
    validation_parts = []
    test_parts = []

    # Split each Potability class separately so every split stays balanced.
    for label, group in dataframe.groupby("Potability", sort=True):
        shuffled_group = group.sample(
            frac=1,
            random_state=SEED + int(label)
        ).reset_index(drop=True)

        total_rows = len(shuffled_group)
        test_rows = round(total_rows * TEST_RATIO)
        validation_rows = round(total_rows * VALIDATION_RATIO)

        test_parts.append(shuffled_group.iloc[:test_rows])
        validation_parts.append(
            shuffled_group.iloc[test_rows:test_rows + validation_rows]
        )
        train_parts.append(
            shuffled_group.iloc[test_rows + validation_rows:]
        )

    train_data = pd.concat(train_parts).sample(
        frac=1, random_state=SEED
    ).reset_index(drop=True)

    validation_data = pd.concat(validation_parts).sample(
        frac=1, random_state=SEED
    ).reset_index(drop=True)

    test_data = pd.concat(test_parts).sample(
        frac=1, random_state=SEED
    ).reset_index(drop=True)

    return train_data, validation_data, test_data


In [25]:
train_df, validation_df, test_df = create_stratified_splits(clean_df)

# Safety check: no row should be lost during splitting.
assert len(train_df) + len(validation_df) + len(test_df) == len(clean_df)

# Create the folder where processed, usable data belongs.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save the cleaned full dataset and the three reproducible splits.
clean_df.to_csv(PROCESSED_DIR / "water_potability_clean.csv", index=False)
train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
validation_df.to_csv(PROCESSED_DIR / "validation.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)

print("DATA SPLIT AND SAVE SUMMARY")
print("-" * 45)
print(f"Cleaned dataset:    {len(clean_df)} rows")
print(f"Training dataset:   {len(train_df)} rows")
print(f"Validation dataset: {len(validation_df)} rows")
print(f"Test dataset:       {len(test_df)} rows")
print(f"\nRandom seed used: {SEED}")

DATA SPLIT AND SAVE SUMMARY
---------------------------------------------
Cleaned dataset:    2556 rows
Training dataset:   1788 rows
Validation dataset: 384 rows
Test dataset:       384 rows

Random seed used: 42


In [26]:
print("\nClass distribution in each split:")
for split_name, split_data in {
    "Training": train_df,
    "Validation": validation_df,
    "Test": test_df,
}.items():
    print(f"\n{split_name}:")
    print(split_data['Potability'].value_counts().sort_index())

print("\nSaved files:")
for file_path in sorted(PROCESSED_DIR.glob("*.csv")):
    print("-", file_path.name)


Class distribution in each split:

Training:
Potability
0    894
1    894
Name: count, dtype: int64

Validation:
Potability
0    192
1    192
Name: count, dtype: int64

Test:
Potability
0    192
1    192
Name: count, dtype: int64

Saved files:
- test.csv
- train.csv
- validation.csv
- water_potability_clean.csv


## Generated Testing Scenarios

The cited Water Potability dataset contains 2,556 real records.

For this SC11 teaching example, we create 256 generated testing scenarios. This is 10% of the real dataset.

These scenarios are only for product testing and demonstration. They are not new laboratory observations and must never be mixed into training, validation, or final test data.

In [27]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from generate_test_scenarios import run_generator

run_generator()

scenarios = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "generated"
    / "generated_water_quality_test_scenarios.csv"
)

print("Scenario table shape:", scenarios.shape)

print("\nExpected class balance:")
print(scenarios["Expected_Potability"].value_counts().sort_index())

print("\nFirst five generated testing scenarios:")
display(scenarios.head())

SC11 GENERATED TESTING SCENARIOS COMPLETED
--------------------------------------------------
Real cleaned records: 2556
Generated testing scenarios: 256
Generation seed: 42
Output file: D:\sample_project\SC11-Water-Quality-Potability-Example\data\generated\generated_water_quality_test_scenarios.csv
Report file: D:\sample_project\SC11-Water-Quality-Potability-Example\data\reports\generated_scenarios_report.md
Scenario table shape: (256, 13)

Expected class balance:
Expected_Potability
0    128
1    128
Name: count, dtype: int64

First five generated testing scenarios:


,Scenario_ID,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Expected_Potability,Scenario_Type,Generation_Seed
0,GEN-WQ-001,8.723150,213.541215,18849.482843,8.684088,323.983758,394.470004,20.070274,96.538713,4.563208,0,generated_testing_only,42
1,GEN-WQ-002,8.855924,223.058111,19396.677272,5.709123,395.766086,466.221103,10.409643,73.827611,3.475455,0,generated_testing_only,42
2,GEN-WQ-003,5.614211,204.526839,16678.920204,5.590040,315.993728,342.099722,15.212163,73.402687,3.469140,0,generated_testing_only,42
3,GEN-WQ-004,5.587951,188.626234,28616.778088,7.556170,326.163806,277.390151,8.373187,54.737168,2.569132,0,generated_testing_only,42
4,GEN-WQ-005,8.165184,201.829737,25721.178228,7.631736,367.194996,597.524816,11.807852,72.546282,3.703196,0,generated_testing_only,42
